# Pysqldb3 tables_created bug

Problem: Seth got an error that the tables_created function would not return the tables he created. Seems like it's not recording them. The goal is for me to emulate the error so we can fix it.

In [1]:
from pysqldb3 import pysqldb3
import os

username = os.environ.get('postgres_username')
password_p = os.environ.get('postgres_password')
db = pysqldb3.DbConnect(type = 'PG',server = 'dotdevrhpgsql02', database = 'ris3', 
                            allow_temp_tables=True, user= username, password= password_p,ldap=False, port=5432, use_native_driver=True, default=False, quiet=False)

Step 1. Create function that automates table creation and deletion so we can experiment with bulk creation/deletion

In [2]:
def create_dummy_table(id_num):

    db.query(f"""
           DROP TABLE IF EXISTS staging.test_table_{id_num};
           CREATE TABLE IF NOT EXISTS staging.test_table_{id_num}
           (
               shape_id character varying,
               shape_pt_lat double precision,
               shape_pt_lon double precision,
               shape_pt_sequence bigint,
               geom geometry(Point,2263)
           );
           CREATE INDEX IF NOT EXISTS test_table_id_{id_num}
           ON staging.test_table_{id_num} USING btree (shape_id);

           CREATE INDEX IF NOT EXISTS test_table_geom_{id_num}
           ON staging.test_table_{id_num} USING gist (geom);
       """)
    
def drop_dummy_table(id_num):

    db.drop_table(schema = 'staging', table = f'test_table_{id_num}')

Step 2. Create a large number of tables and check tables_created

In [17]:
for i in range(1, 15):
    create_dummy_table(i)

db.tables_created

- Query run 2026-05-29 10:42:15.085361
 Query time: Query run in 9149 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.152076
 Query time: Query run in 0 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.237781
 Query time: Query run in 17744 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.300271
 Query time: Query run in 2984 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.392422
 Query time: Query run in 11282 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.453902
 Query time: Query run in 15401 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.520790
 Query time: Query run in 15607 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.590530
 Query time: Query run in 4498 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.668753
 Query time: Query run in 0 microseconds 
 * Returned 0 rows *
- Query run 2026-05-29 10:42:15.737648
 Query time: Query

[(None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_5'),
 (None, None, 'staging', 'test_table_6'),
 (None, None, 'staging', 'test_table_7'),
 (None, None, 'staging', 'test_tab

Step 3. Try a combination of creation and deletion to see if at some point the tables start beging properly recorded

In [5]:
for i in range(1, 15):
    drop_dummy_table(i)
    create_dummy_table(i)

db.tables_created

[(None, None, 'staging', 'test_table_1'),
 (None, None, 'staging', 'test_table_2'),
 (None, None, 'staging', 'test_table_3'),
 (None, None, 'staging', 'test_table_4'),
 (None, None, 'staging', 'test_table_5'),
 (None, None, 'staging', 'test_table_6'),
 (None, None, 'staging', 'test_table_7'),
 (None, None, 'staging', 'test_table_8'),
 (None, None, 'staging', 'test_table_9'),
 (None, None, 'staging', 'test_table_10'),
 (None, None, 'staging', 'test_table_11'),
 (None, None, 'staging', 'test_table_12'),
 (None, None, 'staging', 'test_table_13'),
 (None, None, 'staging', 'test_table_14')]